# 🏃 CardioGoodFitness — Exploratory Data Analysis

**Dataset:** `cardiogoodfitness.csv`  
**Objective:** Build customer profiles for each treadmill model and generate business insights.

---

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Global style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

print('Libraries loaded ✅')

In [ ]:
df = pd.read_csv('cardiogoodfitness.csv')
print('Shape:', df.shape)
df.head()

## 2. Data Overview

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Duplicate Rows ===')
print(df.duplicated().sum())

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues').format(precision=2)

---
## Q1. How many models does the store have?

In [ ]:
models = df['Product'].unique()
print(f'Number of models: {len(models)}')
print(f'Models: {sorted(models)}')

---
## Q2. Which is the most sold model?

In [ ]:
model_counts = df['Product'].value_counts().reset_index()
model_counts.columns = ['Product', 'Count']
model_counts['Percentage'] = (model_counts['Count'] / len(df) * 100).round(2)
print(model_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
bars = axes[0].bar(model_counts['Product'], model_counts['Count'],
                   color=sns.color_palette('Set2', len(models)), edgecolor='white', linewidth=1.5)
for bar, pct in zip(bars, model_counts['Percentage']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{pct:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Units Sold per Model', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Product Model')
axes[0].set_ylabel('Number of Customers')

# Pie chart
axes[1].pie(model_counts['Count'], labels=model_counts['Product'],
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('Set2', len(models)))
axes[1].set_title('Market Share by Model', fontsize=14, fontweight='bold')

plt.suptitle('Q2: Most Sold Treadmill Model', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(f"\n✅ Most sold model: {model_counts.iloc[0]['Product']} with {model_counts.iloc[0]['Count']} units ({model_counts.iloc[0]['Percentage']}%)")

---
## Q3. Are Male customers buying treadmills more than Female customers?

In [ ]:
gender_counts = df['Gender'].value_counts()
print(gender_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Overall gender split
axes[0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452'], startangle=90, explode=[0.05]*len(gender_counts))
axes[0].set_title('Overall Gender Split', fontsize=13, fontweight='bold')

# Gender by product
gender_product = df.groupby(['Product', 'Gender']).size().unstack(fill_value=0)
gender_product.plot(kind='bar', ax=axes[1], color=['#DD8452', '#4C72B0'],
                    edgecolor='white', linewidth=1.2)
axes[1].set_title('Gender Distribution by Product', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Product Model')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Gender')

plt.suptitle('Q3: Gender vs Treadmill Purchase', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Q4. Income, Age & Education of customers buying treadmills

In [ ]:
print('=== Overall Statistics ===')
print(df[['Age', 'Education', 'Income']].describe().round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cols = ['Age', 'Education', 'Income']
colors = ['#4C72B0', '#55A868', '#C44E52']

for ax, col, color in zip(axes, cols, colors):
    ax.hist(df[col], bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='red', linestyle=':', linewidth=1.5, label=f'Median: {df[col].median():.1f}')
    ax.set_title(f'Distribution of {col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('Q4: Age, Education & Income Distributions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Q5. How many days and miles do customers expect to use the treadmill?

In [ ]:
print('Usage (days/week):'); print(df['Usage'].describe().round(2))
print('\nMiles expected:');   print(df['Miles'].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Usage frequency
usage_counts = df['Usage'].value_counts().sort_index()
axes[0].bar(usage_counts.index.astype(str), usage_counts.values,
            color=sns.color_palette('Blues_d', len(usage_counts)), edgecolor='white')
axes[0].set_title('Expected Weekly Usage (days)', fontweight='bold')
axes[0].set_xlabel('Days per Week')
axes[0].set_ylabel('Count')

# Miles
axes[1].hist(df['Miles'], bins=25, color='#55A868', edgecolor='white', alpha=0.85)
axes[1].axvline(df['Miles'].mean(), color='red', linestyle='--', linewidth=2,
                label=f"Mean: {df['Miles'].mean():.1f}")
axes[1].set_title('Expected Miles to Run', fontweight='bold')
axes[1].set_xlabel('Miles')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Q5: Usage Frequency & Miles Expected', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Q6. Self-rated fitness of customers

In [ ]:
fitness_counts = df['Fitness'].value_counts().sort_index()
print(fitness_counts)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(fitness_counts.index.astype(str), fitness_counts.values,
              color=sns.color_palette('RdYlGn', 5), edgecolor='white', linewidth=1.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', fontweight='bold')
ax.set_title('Q6: Self-Rated Fitness Score Distribution\n(1 = Very Unfit → 5 = Very Fit)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Fitness Score')
ax.set_ylabel('Number of Customers')
ax.set_xticks(range(1, 6))
ax.set_xticklabels(['1\n(Very Unfit)', '2', '3\n(Average)', '4', '5\n(Very Fit)'])
plt.tight_layout()
plt.show()

---
## Q7. Are married customers buying more than single customers?

In [ ]:
ms_counts = df['MaritalStatus'].value_counts()
print(ms_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(ms_counts, labels=ms_counts.index, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452'], startangle=90)
axes[0].set_title('Overall Marital Status Split', fontweight='bold')

ms_product = df.groupby(['Product', 'MaritalStatus']).size().unstack(fill_value=0)
ms_product.plot(kind='bar', ax=axes[1], color=['#4C72B0', '#DD8452'],
                edgecolor='white', linewidth=1.2)
axes[1].set_title('Marital Status by Product', fontweight='bold')
axes[1].set_xlabel('Product Model')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Marital Status')

plt.suptitle('Q7: Marital Status vs Treadmill Purchase', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Q8. Relation between Income and Model

In [ ]:
income_by_model = df.groupby('Product')['Income'].describe().round(2)
print(income_by_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Product', y='Income', palette='Set2', ax=axes[0])
axes[0].set_title('Income Distribution by Model', fontweight='bold')
axes[0].set_xlabel('Product Model')
axes[0].set_ylabel('Income ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

sns.violinplot(data=df, x='Product', y='Income', palette='Set2', inner='quartile', ax=axes[1])
axes[1].set_title('Income Density by Model (Violin)', fontweight='bold')
axes[1].set_xlabel('Product Model')
axes[1].set_ylabel('Income ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Q8: Income vs Product Model', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Q9. Relation between Age and Model

In [ ]:
age_by_model = df.groupby('Product')['Age'].describe().round(2)
print(age_by_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Product', y='Age', palette='Set2', ax=axes[0])
axes[0].set_title('Age Distribution by Model', fontweight='bold')
axes[0].set_xlabel('Product Model')
axes[0].set_ylabel('Age (years)')

for product in sorted(df['Product'].unique()):
    subset = df[df['Product'] == product]['Age']
    axes[1].hist(subset, bins=15, alpha=0.6, label=product)
axes[1].set_title('Age Distribution Overlap by Model', fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')
axes[1].legend(title='Model')

plt.suptitle('Q9: Age vs Product Model', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Q10. Relation between Self-Rated Fitness and Model

In [ ]:
fitness_product = df.groupby(['Product', 'Fitness']).size().unstack(fill_value=0)
print(fitness_product)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fitness_product.plot(kind='bar', ax=axes[0], colormap='RdYlGn', edgecolor='white')
axes[0].set_title('Fitness Score Counts by Model', fontweight='bold')
axes[0].set_xlabel('Product Model')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Fitness Score', loc='upper right')

sns.boxplot(data=df, x='Product', y='Fitness', palette='Set2', ax=axes[1])
axes[1].set_title('Fitness Score Distribution by Model', fontweight='bold')
axes[1].set_xlabel('Product Model')
axes[1].set_ylabel('Fitness Score')
axes[1].set_yticks(range(1, 6))

plt.suptitle('Q10: Self-Rated Fitness vs Product Model', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Q11. Relation between Education and Model

In [ ]:
edu_by_model = df.groupby('Product')['Education'].describe().round(2)
print(edu_by_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Product', y='Education', palette='Set2', ax=axes[0])
axes[0].set_title('Education (years) by Model — Boxplot', fontweight='bold')
axes[0].set_xlabel('Product Model')
axes[0].set_ylabel('Education (years)')

sns.stripplot(data=df, x='Product', y='Education', palette='Set2',
              jitter=True, alpha=0.5, ax=axes[1])
axes[1].set_title('Education by Model — Stripplot', fontweight='bold')
axes[1].set_xlabel('Product Model')
axes[1].set_ylabel('Education (years)')

plt.suptitle('Q11: Education Level vs Product Model', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Q12. Does Gender affect which model a customer buys?

In [ ]:
gender_model = pd.crosstab(df['Product'], df['Gender'])
gender_model_pct = pd.crosstab(df['Product'], df['Gender'], normalize='index').round(3) * 100
print('Counts:'); print(gender_model)
print('\nRow %:'); print(gender_model_pct)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gender_model.plot(kind='bar', ax=axes[0], color=['#DD8452', '#4C72B0'],
                  edgecolor='white', linewidth=1.2)
axes[0].set_title('Gender Count by Model', fontweight='bold')
axes[0].set_xlabel('Product Model')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

gender_model_pct.plot(kind='bar', stacked=True, ax=axes[1],
                      color=['#DD8452', '#4C72B0'], edgecolor='white')
axes[1].set_title('Gender % by Model (Stacked)', fontweight='bold')
axes[1].set_xlabel('Product Model')
axes[1].set_ylabel('Percentage')
axes[1].tick_params(axis='x', rotation=0)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())

plt.suptitle('Q12: Gender Effect on Model Choice', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Chi-Square Test
chi2, p, dof, _ = stats.chi2_contingency(gender_model)
print(f'\nChi-Square Test: χ²={chi2:.3f}, p-value={p:.4f}, dof={dof}')
if p < 0.05:
    print('✅ Significant association between Gender and Model (p < 0.05)')
else:
    print('❌ No significant association between Gender and Model (p ≥ 0.05)')

---
## Q13. Does Marital Status affect which model a customer buys?

In [ ]:
ms_model = pd.crosstab(df['Product'], df['MaritalStatus'])
ms_model_pct = pd.crosstab(df['Product'], df['MaritalStatus'], normalize='index').round(3) * 100
print('Counts:'); print(ms_model)
print('\nRow %:'); print(ms_model_pct)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ms_model.plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'],
              edgecolor='white', linewidth=1.2)
axes[0].set_title('Marital Status Count by Model', fontweight='bold')
axes[0].set_xlabel('Product Model')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

ms_model_pct.plot(kind='bar', stacked=True, ax=axes[1],
                  color=['#4C72B0', '#DD8452'], edgecolor='white')
axes[1].set_title('Marital Status % by Model (Stacked)', fontweight='bold')
axes[1].set_xlabel('Product Model')
axes[1].set_ylabel('Percentage')
axes[1].tick_params(axis='x', rotation=0)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())

plt.suptitle('Q13: Marital Status Effect on Model Choice', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

chi2, p, dof, _ = stats.chi2_contingency(ms_model)
print(f'\nChi-Square Test: χ²={chi2:.3f}, p-value={p:.4f}')
print('✅ Significant' if p < 0.05 else '❌ Not significant')

---
## Q14. Are different age groups buying different models?

In [ ]:
bins = [0, 25, 35, 45, 100]
labels = ['18–25', '26–35', '36–45', '45+']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

age_model = pd.crosstab(df['AgeGroup'], df['Product'])
print(age_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

age_model.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white')
axes[0].set_title('Age Group vs Model Purchases', fontweight='bold')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Product')

age_model_pct = age_model.div(age_model.sum(axis=1), axis=0) * 100
age_model_pct.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', edgecolor='white')
axes[1].set_title('Age Group Model Share (Stacked %)', fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Percentage')
axes[1].tick_params(axis='x', rotation=0)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].legend(title='Product')

plt.suptitle('Q14: Age Groups and Model Preferences', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Q15. Relation between Age, Income, Education and Model Bought

In [ ]:
# Pairplot
g = sns.pairplot(df[['Age', 'Income', 'Education', 'Product']],
                 hue='Product', palette='Set2', diag_kind='kde',
                 plot_kws={'alpha': 0.6})
g.fig.suptitle('Q15: Pairplot — Age, Income, Education by Model',
               fontsize=14, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# Heatmap of mean values per model
profile = df.groupby('Product')[['Age', 'Education', 'Income', 'Fitness', 'Usage', 'Miles']].mean().round(2)
print(profile)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(profile.T, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Customer Profile Heatmap (Mean Values per Model)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Product Model')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

---
## 🔑 Correlation Heatmap (All Numeric Features)

In [ ]:
numeric_cols = ['Age', 'Education', 'Usage', 'Fitness', 'Income', 'Miles']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix — All Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📋 Customer Profile Summary

In [ ]:
profile_full = df.groupby('Product').agg(
    Customers=('Product', 'count'),
    Avg_Age=('Age', 'mean'),
    Avg_Education=('Education', 'mean'),
    Avg_Income=('Income', 'mean'),
    Avg_Fitness=('Fitness', 'mean'),
    Avg_Usage_days=('Usage', 'mean'),
    Avg_Miles=('Miles', 'mean'),
    Male_pct=('Gender', lambda x: (x=='Male').mean()*100),
    Partnered_pct=('MaritalStatus', lambda x: (x=='Partnered').mean()*100)
).round(1)

profile_full.style \
    .background_gradient(cmap='Blues', subset=['Avg_Income', 'Avg_Miles']) \
    .background_gradient(cmap='Oranges', subset=['Avg_Fitness', 'Avg_Usage_days']) \
    .format({'Avg_Income': '${:,.0f}', 'Male_pct': '{:.0f}%', 'Partnered_pct': '{:.0f}%'}) \
    .set_caption('📊 Comprehensive Customer Profile by Treadmill Model')

---
## 💡 Key Insights & Business Recommendations

### Insights

1. **TM195 is the best-seller** — the entry-level model dominates sales (~44%), attracting budget-conscious and fitness-curious customers.
2. **TM798 targets a premium niche** — highest average income, fitness score, expected mileage, and usage frequency. These are serious athletes.
3. **TM498 is the mid-range bridge** — sits between TM195 and TM798 on all metrics, catering to moderately fit, middle-income customers.
4. **Males dominate purchases** across all models, especially TM798 — gender gap is most pronounced in the premium segment.
5. **Married/Partnered customers** form the majority of buyers, suggesting household income may drive upgrades to mid/premium models.
6. **Strong positive correlations** exist between Income ↔ Miles, Fitness ↔ Usage, and Education ↔ Income — all reinforcing the luxury buyer profile.
7. **Age is relatively flat** across models (mid-20s to mid-30s), but TM798 skews slightly older (late 20s).

### Recommendations

| Target | Recommendation |
|--------|----------------|
| **TM195** | Market to first-time buyers and young adults (18–28). Focus on affordability, ease of use, and health-starter messaging. |
| **TM498** | Target working professionals (28–40) with moderate fitness goals. Highlight durability, programs, and value-for-money. |
| **TM798** | Premium campaign targeting high-income, fitness-enthusiast males. Emphasize performance specs, serious training, and exclusivity. |
| **Female Segment** | Under-represented, especially in TM498/TM798. A targeted women's campaign (community, body-positive fitness) could grow this segment. |
| **Upselling** | Customers with high fitness scores on TM195 are natural upsell targets — offer upgrade incentives or loyalty programs. |